# Reflexion Architecture — LangChain

**Pattern:** Generator → Critic → Conditional Rewriter (Python loop)

```
generator_chain
      │ draft
      ▼
critic_chain ──score < 7──▶ rewriter_chain ──▶ critic_chain (retry)
      │
   score >= 7
      ▼
   Output
```

**LangChain approach:** The loop is plain Python — `for attempt in range(MAX_RETRIES)`.  
The critic parses to `SCORE:n VERDICT:PASS/REVISE` format for reliable extraction.

In [ ]:
import os, re
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "Set GOOGLE_API_KEY in .env"
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)
parser = StrOutputParser()
THRESHOLD = 7; MAX_RETRIES = 2
print("✓ Ready")

In [ ]:
generator_chain = (ChatPromptTemplate.from_messages([
    ("system","You are a travel writer."),
    ("human","Write a travel recommendation for {destination}. Include: top attractions, best time, safety tips, weather.")]) | llm | parser)

critic_chain = (ChatPromptTemplate.from_messages([
    ("system","Score 1-10 on: specificity(3pts), weather(2pts), safety(2pts), detail(3pts).\nFormat: SCORE:[n]\nISSUES:[list]\nVERDICT:[PASS/REVISE]"),
    ("human","Recommendation:\n{draft}\n\nScore:")]) | llm | parser)

rewriter_chain = (ChatPromptTemplate.from_messages([
    ("system","Rewrite the recommendation addressing all listed issues."),
    ("human","Original:\n{draft}\n\nIssues:\n{issues}\n\nRewrite:")]) | llm | parser)

print("3 chains ready")

In [ ]:
def parse_critique(text):
    score_m = re.search(r"SCORE:(\d+)", text)
    issues_m = re.search(r"ISSUES:(.+?)(?:\n|$)", text)
    verdict_m = re.search(r"VERDICT:(PASS|REVISE)", text)
    return (int(score_m.group(1)) if score_m else 5,
            issues_m.group(1).strip() if issues_m else "Unknown",
            verdict_m.group(1) if verdict_m else "REVISE")

def run_reflexion(destination):
    draft = generator_chain.invoke({"destination": destination})
    print(f"[Generator] draft {len(draft)} chars")
    for attempt in range(MAX_RETRIES + 1):
        critique = critic_chain.invoke({"draft": draft})
        score, issues, verdict = parse_critique(critique)
        print(f"[Critic] attempt {attempt+1}: score={score}/10 verdict={verdict}")
        if verdict == "PASS" or score >= THRESHOLD:
            print(f"✓ Passed at attempt {attempt+1}")
            break
        if attempt < MAX_RETRIES:
            draft = rewriter_chain.invoke({"draft": draft, "issues": issues})
            print(f"[Rewriter] improved draft")
        else:
            print(f"Max retries — using best draft (score {score}/10)")
    return draft, score

final, score = run_reflexion("Tokyo")
print(f"\nFinal score: {score}/10")
print(final[:500] + "...")

## Key Takeaways

| What You Saw | Why It Matters |
|---|---|
| `SCORE:[n] VERDICT:[PASS/REVISE]` format | Reliable critic output parsing via regex |
| Python `for` loop | Simple, explicit — loop visible in code |
| `MAX_RETRIES` guard | Prevents infinite loops |

**LangChain limitation:** The loop is invisible to observability tools — you need to add logging manually.

### Next: [LangGraph Reflexion →](../LangGraph/reflexion.ipynb)